In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Step-by-Step Version

In [11]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

B, T, C = 5, 16, 32
E = 4  # num experts
K = 2  # top_k

router = nn.Linear(C, E, bias=False)
experts_up = nn.ModuleList([nn.Linear(C, C*4, bias=False) for _ in range(E)])
experts_down = nn.ModuleList([nn.Linear(C*4, C, bias=False) for _ in range(E)])

In [12]:
x = torch.randn(B, T, C)

In [13]:
x_flat = x.reshape(-1, C)          # B*T, C
logits = router(x_flat)       # B*T, E

In [14]:
weights = torch.sigmoid(logits)    # B*T, E

In [15]:
values, indices = torch.topk(weights, K, dim=-1)     # B*T, K

In [16]:
x_flat_stacked = torch.stack([x_flat]*K, dim=1)      # B*T, K, C

In [17]:
x_flat_stacked_flat = x_flat_stacked.reshape(-1, C)  # B*T*K, C

In [18]:
indices_flat = indices.reshape(-1)                   # B*T*K

In [19]:
indices_flat_sorted_indices = torch.argsort(indices_flat, stable=True)  # B*T*K

In [20]:
x_flat_stacked_flat_sorted = x_flat_stacked_flat[indices_flat_sorted_indices]  # B*T*K, C

In [21]:
start_idx = 0
outs = []
for i in range(E):
    num_expert = (indices==i).sum().item()
    end_idx = start_idx + num_expert
    h = experts_up[i](x_flat_stacked_flat_sorted[start_idx:end_idx])
    z = F.relu(h).square()
    o = experts_down[i](z)
    outs.append(o)
    start_idx += num_expert
out_flat_stacked_flat_sorted = torch.cat(outs)   # B*T*K, C

out_flat_stacked_flat = torch.zeros(B*T*K, C, device=x.device, dtype=x.dtype)
out_flat_stacked_flat[indices_flat_sorted_indices] = out_flat_stacked_flat_sorted   # B*T*K, C
out_flat_stacked = out_flat_stacked_flat.reshape(B*T, K, C)
out_flat_stacked_weighted = out_flat_stacked * values.unsqueeze(-1)
out_flat = out_flat_stacked_weighted.sum(dim=1)   # B*T, C
outputs = out_flat.reshape(B, T, C)

In [22]:
outputs[0, :8, :6]

tensor([[-0.0663,  0.0811, -0.1379,  0.1524,  0.0222, -0.1163],
        [ 0.2123, -0.0613, -0.0698,  0.0877, -0.1117,  0.1117],
        [ 0.0378, -0.2270, -0.0242, -0.0295,  0.1607, -0.0553],
        [ 0.7392, -0.1019,  0.1579,  0.0130,  0.0290, -0.2059],
        [ 0.1617,  0.0850,  0.0594,  0.0576,  0.0664,  0.0860],
        [-0.2229, -0.0129, -0.2217,  0.2586,  0.1082,  0.1291],
        [-0.1980, -0.0052,  0.0817,  0.4465, -0.4593, -0.1627],
        [-0.0289,  0.1195, -0.0447, -0.0812,  0.1172,  0.0198]],
       grad_fn=<SliceBackward0>)

## Combined Version

In [23]:
class MyMoE(torch.nn.Module):
    def __init__(self, C, E, K):
        super().__init__()
        self.K = K
        self.router = nn.Linear(C, E, bias=False)
        self.experts_up = nn.ModuleList([nn.Linear(C, C*4, bias=False) for _ in range(E)])
        self.experts_down = nn.ModuleList([nn.Linear(C*4, C, bias=False) for _ in range(E)])

    @torch.compiler.disable  # Dynamic slicing breaks the torch.compile
    def forward(self, x):
        B, T, C = x.shape
        K = self.K
        E = len(self.experts_up)
        x_flat = x.reshape(-1, C)          # B*T, C
        logits = self.router(x_flat)       # B*T, E
        weights = torch.sigmoid(logits)    # B*T, E
        values, indices = torch.topk(weights, K, dim=-1)     # B*T, K
        x_flat_stacked = torch.stack([x_flat]*K, dim=1)      # B*T, K, C
        x_flat_stacked_flat = x_flat_stacked.reshape(-1, C)  # B*T*K, C
        indices_flat = indices.reshape(-1)                   # B*T*K
        indices_flat_sorted_indices = torch.argsort(indices_flat, stable=True)  # B*T*K
        x_flat_stacked_flat_sorted = x_flat_stacked_flat[indices_flat_sorted_indices]  # B*T*K, C
        
        start_idx = 0
        outs = []
        for i in range(E):
            num_expert = (indices==i).sum().item()
            end_idx = start_idx + num_expert
            h = self.experts_up[i](x_flat_stacked_flat_sorted[start_idx:end_idx])
            z = F.relu(h).square()
            o = self.experts_down[i](z)
            outs.append(o)
            start_idx += num_expert
        out_flat_stacked_flat_sorted = torch.cat(outs)   # B*T*K, C
        
        out_flat_stacked_flat = torch.zeros(B*T*K, C, device=x.device, dtype=x.dtype)
        out_flat_stacked_flat[indices_flat_sorted_indices] = out_flat_stacked_flat_sorted   # B*T*K, C
        out_flat_stacked = out_flat_stacked_flat.reshape(B*T, K, C)
        out_flat_stacked_weighted = out_flat_stacked * values.unsqueeze(-1)
        out_flat = out_flat_stacked_weighted.sum(dim=1)   # B*T, C
        outputs = out_flat.reshape(B, T, C)
        return outputs

In [24]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

B, T, C = 5, 16, 32
E = 4  # num experts
K = 2  # top_k

moe = MyMoE(C=C, E=E, K=K)

x = torch.randn(B, T, C)
outputs = moe(x)
loss = outputs.float().square().mean()
loss.backward()

In [25]:
outputs[0, :8, :6]

tensor([[-0.0663,  0.0811, -0.1379,  0.1524,  0.0222, -0.1163],
        [ 0.2123, -0.0613, -0.0698,  0.0877, -0.1117,  0.1117],
        [ 0.0378, -0.2270, -0.0242, -0.0295,  0.1607, -0.0553],
        [ 0.7392, -0.1019,  0.1579,  0.0130,  0.0290, -0.2059],
        [ 0.1617,  0.0850,  0.0594,  0.0576,  0.0664,  0.0860],
        [-0.2229, -0.0129, -0.2217,  0.2586,  0.1082,  0.1291],
        [-0.1980, -0.0052,  0.0817,  0.4465, -0.4593, -0.1627],
        [-0.0289,  0.1195, -0.0447, -0.0812,  0.1172,  0.0198]],
       grad_fn=<SliceBackward0>)